# insurance-rag Colab GPU eval

Working notebook assembled from the 2026-09-16 session that first got this running end to end on a T4. See `docs/colab-gpu-plan.md` for the original plan and rationale, and `Challenges and Learnings.md` for the deeper write-ups. This notebook is the practical "run these cells in this order" version.

**Before running anything: Runtime -> Change runtime type -> T4 GPU.**

This run used Qdrant Cloud (not embedded/local mode) and `gemma3:12b` as an independent judge, `llama3.2:3b` then `llama3.1:8b` as generation models, to isolate judge-model and generation-model effects separately against the local CPU baseline (Faithfulness 0.706, see `data/eval/answer_eval_results.md`).

## 1. Clone the repo

In [ ]:
!git clone https://github.com/ashishrahate/insurance-rag.git
%cd insurance-rag

## 2. Install Python deps

Use `requirements-colab.txt`, not `requirements.txt` -- the latter has `pywin32` (Windows-only, aborts the whole install) and pins `torch==2.14.0` from a Windows CPU freeze, which clobbers Colab's preinstalled CUDA-matched `torch`/`torchvision` pair. See the comments at the top of `requirements-colab.txt` for the full story.

In [ ]:
!pip install -r requirements-colab.txt

**Restart the runtime now** (Runtime -> Restart session). Required even on a clean install -- several packages Colab preloads (numpy, httpx, etc.) won't pick up versions installed after they were first imported.

After restarting, re-run the `%cd insurance-rag` cell below (session state resets on restart) before continuing.

In [ ]:
%cd insurance-rag

## 2b. Sanity check: GPU + the transformers/sentence-transformers import chain

This is the step that broke repeatedly in the original session. Run it now, before touching Ollama or Qdrant, so any fix happens early.

In [ ]:
import torch
print(torch.__version__, 'cuda available:', torch.cuda.is_available())

from transformers.modeling_utils import PreTrainedModel
from sentence_transformers import CrossEncoder
print('transformers/sentence-transformers import: ok')

### If the cell above fails

- `torch.cuda.is_available()` is `False`, or torch/torchvision import errors (`RuntimeError: operator torchvision::nms does not exist`, or similar ABI mismatch): Colab's preinstalled `torch`/`torchvision` pair got clobbered. Reinstall the matched pair explicitly (check `pip show torchvision`'s conflict warnings for the exact versions Colab currently ships -- 2.11.0+cu128 / 0.26.0+cu128 was current as of this session, may differ later):
  ```
  !pip install torch==2.11.0+cu128 torchvision==0.26.0+cu128 --index-url https://download.pytorch.org/whl/cu128
  ```
  Then **restart the runtime again** and re-run the `%cd` and import-check cells.

- `ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?` -- this message is misleading; it's transformers' lazy-loader swallowing the real error. Get the real cause by importing the submodule directly instead of through `transformers.__getattr__`:
  ```
  from transformers.modeling_utils import PreTrainedModel
  ```
  This usually surfaces either a missing `accelerate` (already in `requirements-colab.txt`) or the torchvision mismatch above.

## 3. Ollama + models

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
!ollama pull llama3.1:8b
!ollama pull llama3.2:3b
!ollama pull gemma3:12b
!ollama pull nomic-embed-text
!ollama list

## 4. Qdrant Cloud

Sign up at cloud.qdrant.io, create a free-tier cluster, grab its URL (includes `:6333`) and an API key. Don't commit these -- set them as runtime-only variables.

**Must run before any import of `config.settings` or anything from `src/` in this kernel session** -- `config/settings.py` reads env vars once at import time; setting them after an earlier import (even a failed one) silently has no effect until you restart the runtime.

In [ ]:
import os
os.environ['QDRANT_URL'] = 'https://your-cluster-id.your-region.gcp.cloud.qdrant.io:6333'
os.environ['QDRANT_API_KEY'] = 'your-api-key'

In [ ]:
from src.retrieval.search import get_client
client = get_client()
print(client.get_collections())  # expect an empty list, no error

## 5. Ingest into the Cloud collection

`bootstrap_collection` creates the collection and payload indexes, including `doc_id` -- needed because `chunk_and_index.py` filters deletes by `doc_id` for idempotent re-indexing, which Qdrant Cloud's strict mode rejects if `doc_id` isn't indexed (local Docker Qdrant tolerates it unindexed via full scan; Cloud doesn't).

In [ ]:
!python -m src.ingestion.bootstrap_collection

`scrape_ca_bulletins` defaults to `--limit 18`. To reproduce the full local 37-doc corpus, pass the limit explicitly -- otherwise you'll silently get a smaller corpus than your local baseline and the two runs won't be comparable.

In [ ]:
!python -m src.ingestion.scrape_ca_bulletins --limit 37 --min-year 2020
!python -m src.ingestion.parse_ca_bulletins
!python -m src.ingestion.chunk_and_index --naive

In [ ]:
print(client.get_collection('insurance_ca_v1').points_count)  # expect 129 for the 37-doc corpus

## 6. Judge service

Set `JUDGE_LLM_MODEL` before starting the service -- same import-time-env-var rule as `QDRANT_URL` above.

If the Colab runtime gets restarted for any reason (e.g. fixing the torch/torchvision issue above) after this point, both `ollama serve` and this `uvicorn` process die with it and need restarting -- `!ollama list` and the curl check below are the way to notice before wasting a full eval run against a dead service.

In [ ]:
import os
os.environ['JUDGE_LLM_MODEL'] = 'gemma3:12b'
os.environ['LLM_MODEL'] = 'llama3.2:3b'  # first run: same gen model as local baseline, isolates the judge-model effect
os.environ['RUN_ENV'] = 'colab-t4'

In [ ]:
!nohup uvicorn src.judge_service.main:app --port 8100 > judge.log 2>&1 &
!sleep 3
!curl -s http://localhost:8100/docs -o /dev/null -w "%{http_code}\n"  # expect 200
!cat judge.log

## 7. Run the evals

In [ ]:
!python -m src.evaluation.retrieval_eval --retriever hybrid_rerank --label "colab-t4, llama3.2:3b gen, gemma3:12b judge"

In [ ]:
!python -m src.evaluation.run_full_eval --label "colab-t4, llama3.2:3b gen, gemma3:12b judge"

### Second run: bump the generation model, keep the judge fixed

Isolates the generation-model effect on top of the (already-changed) judge, rather than confounding both changes in one run.

In [ ]:
import os
os.environ['LLM_MODEL'] = 'llama3.1:8b'

!python -m src.evaluation.run_full_eval --label "colab-t4, llama3.1:8b gen, gemma3:12b judge"

## 8. Bring results back

In [ ]:
from google.colab import files
files.download('logs/runs.jsonl')
files.download('data/eval/results.md')
files.download('data/eval/answer_eval_results.md')

Locally, compare against the CPU baseline:
```
python -m src.observability.report --file colab_runs.jsonl --group env
```